# Part 1 – Pandas DataFrames

In [ ]:
!pip install pyreadstat
!pip install savReaderWriter

In [ ]:
import numpy as np

In [ ]:
import pandas as pd

In [ ]:
# 1. Importar dataset Educación (Módulo 300)
df300 = pd.read_csv(
    r"/Users/freddyzutachavez/Documents/ENAHO-2023/Enaho01A-2023-300.csv",
    encoding="ISO-8859-1",
    low_memory=False
)


In [ ]:
# 2. Mostrar las primeras 5 filas
print("Primeras 5 filas del dataset:")
display(df300.head())

In [ ]:
# 3. Convertir nombres de columnas a lista
columnas = df300.columns.tolist()
print("\nLista de nombres de columnas:")
print(columnas)

In [ ]:
# 4. Revisar tipos de datos
print("\nTipos de datos del DataFrame:")
print(df300.dtypes)


In [ ]:
# 5. Selección de un subconjunto con variables clave + adicionales
sub300 = df300[['CONGLOME', 'VIVIENDA', 'HOGAR', 'CODPERSO',
                'P301A', 'P301B', 'P302A']]  # <-- aquí puse ejemplos, ajusta a tu interés
print("\nSubmuestra seleccionada:")
display(sub300.head())

In [ ]:
# 6. Exploración del DataFrame
print("\nResumen estadístico:")
print(sub300.describe(include='all'))

In [ ]:
# 7. Identificar valores faltantes
print("\nValores faltantes por columna:")
print(sub300.isnull().sum())

In [ ]:

# 8. Eliminar filas con valores faltantes
sub300_clean = sub300.dropna()
print("\nSubmuestra después de eliminar NA:")
display(sub300_clean.head())

In [ ]:
# ==========================
# Importar librerías
# ==========================
import pandas as pd
import numpy as np

In [ ]:
# ==========================
# Dataset 2: Vivienda (200)
# ==========================
df200 = pd.read_csv(
    r"/Users/freddyzutachavez/Documents/ENAHO-2023/Enaho01-2023-200.csv",
    encoding="ISO-8859-1",
    low_memory=False
)


In [ ]:
print("Primeras 5 filas del dataset 200 (Vivienda):")
display(df200.head())

In [ ]:
print("\nColumnas (200):")
print(df200.columns.tolist())

In [ ]:
print("\nTipos de datos (200):")
print(df200.dtypes)

In [ ]:

# Mostrar las columnas de cada dataset
print("Columnas del primer dataset (Educación - 300):")
print(df300.columns.tolist())

print("\nColumnas del segundo dataset (Vivienda - 200):")
print(df200.columns.tolist())

# 🔎 Identificar las columnas comunes entre ambos
common_columns = list(set(df300.columns).intersection(set(df200.columns)))
print("\nColumnas comunes entre los dos datasets:")
print(common_columns)



## Merge 

In [ ]:

# 1) Definir columnas clave recomendadas para ENAHO
common_columns = ['CONGLOME', 'VIVIENDA', 'HOGAR', 'CODPERSO']

# Verificar que existan en ambos DataFrames
faltan_300 = [c for c in common_columns if c not in df300.columns]
faltan_200 = [c for c in common_columns if c not in df200.columns]
if faltan_300 or faltan_200:
    raise KeyError(f"Faltan columnas: en df300 {faltan_300} | en df200 {faltan_200}")

# 2) Estandarizar y corregir tipos en las claves
def limpiar_claves(df, key_cols):
    df = df.copy()

    # Quitar espacios si vinieran como strings
    for c in key_cols:
        if pd.api.types.is_object_dtype(df[c]):
            df[c] = df[c].astype(str).str.strip().replace({'': np.nan})

    # Convertir a numérico (muchas veces vienen como '123.0' o texto)
    for c in key_cols:
        df[c] = pd.to_numeric(df[c], errors='coerce')  # NaN si no se puede convertir
        # Si quedan floats por decimales .0, pásalos a enteros (Int64 tolera NaN)
        if pd.api.types.is_float_dtype(df[c]):
            # Si hay valores no enteros verdaderos, redondea (ajusta según tu criterio)
            # Aquí asumimos que deben ser enteros:
            df[c] = df[c].round().astype('Int64')
        else:
            df[c] = df[c].astype('Int64')

    # Eliminar filas con NaN en claves (no se pueden unir)
    antes = len(df)
    df = df.dropna(subset=key_cols)
    despues = len(df)

    if antes != despues:
        print(f"Se eliminaron {antes - despues} filas con claves nulas.")

    return df

df300_keys = limpiar_claves(df300, common_columns)
df200_keys = limpiar_claves(df200, common_columns)

# 3) Detectar duplicados de clave (claves no únicas complican el merge 1:1)
dup300 = df300_keys.duplicated(subset=common_columns, keep=False).sum()
dup200 = df200_keys.duplicated(subset=common_columns, keep=False).sum()
print(f"Duplicados de clave en df300: {dup300}")
print(f"Duplicados de clave en df200: {dup200}")

# 4) Anti-joins para ver claves que NO coinciden entre ambos
# Crear tuplas de clave para comparar conjuntos
keys300 = set(map(tuple, df300_keys[common_columns].astype('Int64').values))
keys200 = set(map(tuple, df200_keys[common_columns].astype('Int64').values))

solo_en_300 = keys300 - keys200
solo_en_200 = keys200 - keys300

print(f"Claves presentes solo en df300: {len(solo_en_300)}")
print(f"Claves presentes solo en df200: {len(solo_en_200)}")

# Muestras de las primeras diferencias (si existen)
if solo_en_300:
    print("\nEjemplos de claves solo en df300 (primeras 5):")
    for t in list(solo_en_300)[:5]:
        print(t)

if solo_en_200:
    print("\nEjemplos de claves solo en df200 (primeras 5):")
    for t in list(solo_en_200)[:5]:
        print(t)

# (Opcional) DataFrames con filas no emparejadas - útil para inspeccionar y corregir a mano si hace falta
unmatched_300 = df300_keys.merge(df200_keys[common_columns].drop_duplicates(),
                                 on=common_columns, how='left', indicator=True)
unmatched_300 = unmatched_300[unmatched_300['_merge'] == 'left_only']
print(f"\nFilas en df300 sin match en df200: {len(unmatched_300)}")

unmatched_200 = df200_keys.merge(df300_keys[common_columns].drop_duplicates(),
                                 on=common_columns, how='left', indicator=True)
unmatched_200 = unmatched_200[unmatched_200['_merge'] == 'left_only']
print(f"Filas en df200 sin match en df300: {len(unmatched_200)}")

# 5) Merge final
#    - 'inner' conserva solo las claves comunes (unión más "segura")
#    - 'left' conserva todas las filas de df300 (útil si 300 es el universo principal)
#    - 'outer' conserva todo y ayuda a auditar faltantes
merge_how = 'inner'  # ajusta a 'left' o 'outer' según tu necesidad

df_merged = pd.merge(
    df300_keys,
    df200_keys,
    on=common_columns,
    how=merge_how,
    suffixes=('_300', '_200')
)

print(f"\nResultado del merge ({merge_how}): {df_merged.shape[0]} filas x {df_merged.shape[1]} columnas")
display(df_merged.head())


## Muestra las primeras 5 filas del DataFrame resultante

In [ ]:
# Mostrar las primeras 5 filas del DataFrame resultante
df_merged.head()

## Agrupa los datos por una variable de tu elección usando groupby()

In [ ]:
# Contar el número de registros por estrato
conteo_estrato = df_merged.groupby("ESTRATO_300").size()

print("Número de registros por ESTRATO_300:")
print(conteo_estrato)


## Ingreso_promedio_estrato

In [ ]:
ingreso_promedio_estrato = df_merged.groupby("ESTRATO_300")["FACPOB07"].mean().reset_index()
ingreso_promedio_estrato = ingreso_promedio_estrato.rename(columns={"FACPOB07": "Ingreso_Promedio"})

print("Ingreso promedio por ESTRATO:")
print(ingreso_promedio_estrato)

# Parte 2 - If Conditions

## Basic If Condition

In [ ]:
def check_number(x):
    if x > 0:
        return f"The number {x} is positive."
    else:
        return f"The number {x} is not positive."

check_number(4)

In [ ]:
check_number(-1)

## If Condition with Multiple Expressions

In [ ]:
def check_temperature(temp):
    if temp < 0:
        return "Clima muy frío"
    elif temp < 20:
        return "Clima fresco"
    elif temp < 30:
        return "Clima agradable"
    else:
        return "Clima caluroso"

In [ ]:
print(check_temperature(-5))

In [ ]:
print(check_temperature(36))

## Logical Operators

In [ ]:
def scholarship_eligibility(gpa, extracurricular, community_hours):
    if gpa > 3.5 and (extracurricular == "Yes" or community_hours > 50):
        return "Eligible for scholarship."
    else:
        return "Not eligible for scholarship."

In [ ]:
print(scholarship_eligibility(3.8, "Yes", 10)) 

In [ ]:
print(scholarship_eligibility(3.2, "Yes", 60))

## Python Identity Operators

In [ ]:
list1 = [1, 2, 3]
list2 = [1, 2, 3]
list3 = list1

In [ ]:
print("list1 is list2:", list1 is list2)

In [ ]:
print("list1 is list3:", list1 is list3)

In [ ]:
print("list1 == list2:", list1 == list2)

## Nested If Statement

In [ ]:
def assign_grade(score):
    if score >= 90:
        return "A"
    elif 80 <= score <= 89:
        if score == 85:
            return "B+"
        else:
            return "B"
    elif 70 <= score <= 79:
        return "C"
    else:
        return "Fail"

In [ ]:
print(assign_grade(92))

In [ ]:
print(assign_grade(85))

In [ ]:
print(assign_grade(83)) 

In [ ]:
print(assign_grade(75)) 

In [ ]:
print(assign_grade(60))

In [ ]:
print(assign_grade(40))

## Part 3

11) For Loop in NumPy

In [ ]:
array1 = np.array([10,20,30,40,50])
print(array1)

In [ ]:
for x in array1:
    print(x*2)

In [ ]:
Re-question:

In [ ]:
for i, x in enumerate(array1):
    if i == 0:
        result = []
    result.append(x * 2)
    if i == len(array1)-1:
        new_arr=np.array(result)
        print(new_arr)

12) For Loop in List

In [ ]:
wordlist=["python", "loop", "list", "iteration"]

In [ ]:
wordlist

In [ ]:
for e in wordlist:
    print(len(e))

Re-question:

In [ ]:
newlist = [len(e) for e in wordlist]
newlist

13) For Loop in Dictionary

In [ ]:
scores = {"Alice": 85, "Bob": 92, "Charlie": 78, "Diana": 88}

for Students, Grades in scores.items():
    print(Students, Grades)

Re-question

In [ ]:
for Students, Grades in scores.items():
    if Grades > 80:
        print(Students)

In [ ]:
14) For Loop using Range

In [ ]:
for x in range(1, 21):
    if x % 2 == 0:
        print(x)

Re-question

In [ ]:
sumran = 0

for x in range(1, 21):
    if x % 2 == 0:
        sumran = sumran+x
        print(sumran)

15) Iterations over Pandas (ENAHO dataset)
Suppose you are analyzing the National Household Survey (ENAHO) dataset, specifically the file ENAHO01A-2023-400.
The question of interest is P41601: “¿Cuánto fue el monto total por la compra o servicio?”.

Write a for loop that iterates over the column P41601 and prints values greater than 5000.

Re-question: How would you optimize this task using pandas vectorized operations (e.g., boolean indexing) instead of a for loop, to make the analysis faster and more efficient?


In [ ]:
ena400 = pd.read_csv(r"..\group_03_assignment2_2025\Enaho01A-2023-400.csv", encoding="ISO-8859-10", low_memory=False)

In [ ]:
display(ena400.head())

In [ ]:
for que15 in ena400["P41601"]:
    if pd.notna(que15):
        try:
            if float(que15) > 5000:
                print(que15)
        except ValueError:
            pass

Re-question

In [ ]:
ena400["P41601"] = pd.to_numeric(ena400["P41601"])

over5000 = ena400.loc[ena400["P41601"] > 5000, "P41601"]

print(over5000)